# Section 1 Assignment - Transformers

# Task
You work for a large company and the company receives 8 letters regarding one of your products. These letters range from letters of complaint to letters of praise.


Using Hugging Face Transformers,
1. Summarize the sentiments towards the product and
1. Provide a summary of the letters that reflect the most extreme sentiments.

1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the positive responses and negative responses.
1. Determine and report in each case on:
    - What the customer wants to happen by automatically questioning their respective letters.
1. Knowing how each customer feels,
    - Automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

1. In your code to generate the report for your manager and the letters of reply,
    - indicate your design choices and
    - use any pre-processing trick that you anticipate will improve your automatic generation.


## Tools for the task
* Some great working examples can be found in the file **01_introduction.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* You will need to generate/provide the letters of complaint. You can generate the letters from the customers manually or automatically. You can use a Hugging Face text generation pipeline to do this or an online generator like [You.com](https://you.com/search?q=how+to+write+well&&tbm=youwrite&cfr=write&).
* You can use the library such as [python-docx](https://python-docx.readthedocs.io/en/latest/index.html#) to generate the report and the response letters.
* There is a variety of models on Hugging Face for the different tasks, e.g, [question-answering](https://huggingface.co/models?pipeline_tag=question-answering&sort=downloads) that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.

## Environment setup

In [ ]:
#!pip install -q transformers torch sentencepiece

# get rid of widget error
#!pip install nbstripout
#!nbstripout /content/MN5162_Section1_Assignment_gorourke.ipynb

In [1]:
from transformers import pipeline
import pandas as pd
import textwrap

## Class to hold data as we process the feedback

In [29]:
from dataclasses import dataclass
from typing import Any

@dataclass
class CustomerFeedback:
    customer_name: str
    original_letter: str
    normalised_letter:str = None
    summary: str = None
    sentiment: str = None
    user_request: Any = None
    response_letter: str = None

### Build Summarisation Model

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART for summarization
summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [3]:
def summarise_text(model, tokenizer, text):

    # Generate summary
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=150,      # Use max_new_tokens, not max_length
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print("Summary:", summary)

    return summary

## Build Sentiment Model

In [4]:
from transformers import pipeline

classifier = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [5]:
def determine_sentiment(classifier, text):
    outputs = classifier(text)

    for output in outputs:
        label = output["label"]
        score = output["score"]

    return outputs

## Build Question Answering

In [6]:
reader = pipeline("question-answering")


No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def determine_user_request(context) :

    question = "What does the customer want?"

    outputs = reader(question=question, context=context)
    pd.DataFrame([outputs])

    return outputs

## Letter response

In [8]:
generator = pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [53]:
def generate_letter_response(customerFeedback) :

    # prompt = f"""Write a professional customer service reply.  Include their name in the greeting in the response.
    #
    # Customer Name: {customerFeedback.customer_name}
    #
    #Customer Complaint : {customerFeedback.original_letter}
    #
    #Customer Request : {customerFeedback.user_request or "Not Specified"}
    #
    #Reply:
    #"""


    complaint = "I ordered a laptop 2 weeks ago and it still hasn't arrived."

    prompt = f"""You are a professional customer service representative for Acme Electronics,
        a consumer electronics company. Your tone is empathetic, solution-focused, and professional.

        Customer complaint:
        \"\"\"
        {complaint}
        \"\"\"

        Write a response that:
    - Acknowledges the customer's frustration
    - Apologizes sincerely without admitting legal fault
    - Offers a concrete next step or resolution
    - Keeps the response under 150 words

    Do NOT use generic filler phrases like "We value your feedback"
    """



    #, max_length=200
    generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2")
    #outputs = generator(prompt, max_length=1000, do_sample = True, temperature=0.7)

    result = generator(prompt, temperature=0.8, return_full_text=False)
#    outputs = result[0]
    reply = outputs[0]["generated_text"]

    return reply, prompt

## Letter processing

In [26]:
import re

def preprocess_letter_text(text):
    working_text = text.lower()
    working_text = re.sub(r'\s+', ' ', working_text)

    return working_text.strip()

## Main code

In [17]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

In [18]:
sample_letter = text

customer_feedback_records = [
    CustomerFeedback('Bumblebee', text)
]



In [30]:
for feedback in customer_feedback_records:

    feedback.normalised_letter = preprocess_letter_text(feedback.original_letter)

    # generate summary
    summary_text = summarise_text(summary_model, summary_tokenizer, feedback.normalised_letter)
    feedback.summary = textwrap.fill(summary_text, width=80)

    print (summary_text)

    user_request = determine_user_request(feedback.normalised_letter)
    # {'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}
    feedback.user_request = user_request['answer']

    sentiment = determine_sentiment(classifier, feedback.normalised_letter)
    # {'label': 'NEGATIVE', 'score': 0.9015460014343262}
    feedback.sentiment = sentiment

    feedback.response_letter =generate_letter_response(feedback.normalised_letter, feedback.customer_name)

Summary: dear amazon, last week i ordered an optimus prime action figure from your online store in germany. unfortunately, when i opened the package, i discovered to my horror that i had been sent an action figure of megatron instead! as a lifelong enemy of the decepticons, i hope you can understand my dilemma. to resolve the issue, i demand an exchange of Megatron for the optimusprime figure i ordered. enclosed are copies of my records concerning this purchase. i expect to hear from you soon. sincerely, bumblebee.
dear amazon, last week i ordered an optimus prime action figure from your online store in germany. unfortunately, when i opened the package, i discovered to my horror that i had been sent an action figure of megatron instead! as a lifelong enemy of the decepticons, i hope you can understand my dilemma. to resolve the issue, i demand an exchange of Megatron for the optimusprime figure i ordered. enclosed are copies of my records concerning this purchase. i expect to hear from

Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [31]:
from pprint import pprint

for feedback in customer_feedback_records:
    pprint (feedback)

CustomerFeedback(customer_name='Bumblebee',
                 original_letter='Dear Amazon, last week I ordered an Optimus '
                                 'Prime action figure from your online store '
                                 'in Germany. Unfortunately, when I opened the '
                                 'package, I discovered to my horror that I '
                                 'had been sent an action figure of Megatron '
                                 'instead! As a lifelong enemy of the '
                                 'Decepticons, I hope you can understand my '
                                 'dilemma. To resolve the issue, I demand an '
                                 'exchange of Megatron for the Optimus Prime '
                                 'figure I ordered. Enclosed are copies of my '
                                 'records concerning this purchase. I expect '
                                 'to hear from you soon. Sincerely, Bumblebee.',
           

In [43]:
result = generate_letter_response(customer_feedback_records[0])

generated_text = result

print(generated_text)
print ('-' * 50)
response_text = generated_text; #[len(prompt):]

response_formatted = textwrap.fill(response_text, width=80)
print(response_formatted)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Write a customer service response with this context.  Include their name in the greeting in the response.
    
    Customer Name: Bumblebee

    Customer Complaint : Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee.
    
    Customer Request : an exchange of megatron
    
    Reply:
         Reply:
           Reply:
             Reply:               Reply:                                                                                                                                                                                   

In [ ]:
print ( pd.DataFrame(result))

                                      generated_text
0  Write a customer service response with this co...


## Report
1.Provide a summary of the letters that reflect the most extreme sentiments.
1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the positive responses and negative responses.

Determine and report in each case on:

- What the customer wants to happen by automatically questioning their respective letters.
Knowing how each customer feels,

- Automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

In your code to generate the report for your manager and the letters of reply,

indicate your design choices and
use any pre-processing trick that you anticipate will improve your automatic generation.

In [ ]:
# https://python-docx.readthedocs.io/en/latest/index.html#
from docx import Document
from docx.shared import Inches

document = Document()

document.add_heading('Customer Feedback Report', 0)

document.add_heading('Positive Feedback', level = 1)

document.add_heading('Negative Feedback', level = 1)

p = document.add_paragraph('A plain paragraph having some ')
p.add_run('bold').bold = True
p.add_run(' and some ')
p.add_run('italic.').italic = True

document.add_heading('Heading, level 1', level=1)
document.add_paragraph('Intense quote', style='Intense Quote')

document.add_paragraph(
    'first item in unordered list', style='List Bullet'
)
document.add_paragraph(
    'first item in ordered list', style='List Number'
)


## Tests

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

In [ ]:
print(determine_sentiment(classifier, text))

[{'label': 'NEGATIVE', 'score': 0.9015460014343262}]


In [ ]:
print (determine_user_request(text))

{'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}


In [ ]:
summary_format = textwrap.fill(summary_example, width=80)
print(summary_format)

Bumblebee ordered an Optimus Prime action figure from your online store in
Germany. Unfortunately, when I opened the package, I discovered to my horror
that I had been sent an action figure of Megatron instead. As a lifelong enemy
of the Decepticons, I hope you can understand my dilemma. To resolve the issue,
I demand an exchange ofMegatron for the Optimus Prime figure.


In [54]:
results, prompt = generate_letter_response(customer_feedback_records[0])

print(results)
actual_reply = results.replace( prompt, "").strip()

pprint(actual_reply)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/contrib/concurrent.py", line 51, in _executor_map
    return list(tqdm_class(ex.map(fn, *iterables, chunksize=chunksize), **kwargs))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 250, in __iter__
    yield from it
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 619, in result_iterator
    yield _result_or_cancel(fs.pop())
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 317, in _result_or_cancel
    return fut.result(timeout)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 451, in result
    self._condition.wait(timeout)
  File "/usr/lib/python3.12/t

TypeError: object of type 'NoneType' has no len()

# Workings


In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

outputs = summarizer(text, max_length=130, min_length=30, do_sample=False)
#outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
### Letter of complaint
generator = pipeline("text-generation", model="gpt2")

letter_gen_prompt= """
Generate 8 customer letters about an ipad.
Letters 1-4 should complain about the widget.  Vary the level of complaint within these letters ranging from a mild complaint to a serious complaint.
Letters 5-8 should praise the widget.  Vary the level of satisfaction witin these letters ranging from a mild satisfaction to complete satisfaction.

Include a customer name in each letter.

Start each letter with

Dear Customer service,
"""

result = generator(letter_gen_prompt, temperature=0.8)


letters = result[0]["generated_text"].split("Letter")
df = pd.DataFrame(result)
df


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,generated_text
0,\nGenerate 8 customer letters about an ipad. \...


In [ ]:
for index, letter in enumerate(letters):
    print(f'Letter {index+1}:')
    print(f'{letter}')

Letter 1:

Generate 8 customer letters about an ipad. 

Letter 2:
s 1-4 should complain about the widget.  Vary the level of complaint within these letters ranging from a mild complaint to a serious complaint.

Letter 3:
s 5-8 should praise the widget.  Vary the level of satisfaction witin these letters ranging from a mild satisfaction to complete satisfaction.

Include a customer name in each letter.

Start each letter with 

Dear Customer service,

I am wondering if you can send an email address to the customer service team and advise them on if we are happy with the widget. My recommendation would be to send an email to the customer service team to confirm that it is not for my personal use. I would also like to include the name of the widget on the widget.

We would also like to include the name of the widget on the widget.

As an optional bonus, we would like to include the name of the widget on their website.

Since this is only a suggestion, I don't want to include this informat